In [2]:
from pprint import pprint
import json
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
KG_JSON_PATH = os.path.join(PROJECT_ROOT, "data", "knowledge_graph", "test_new.json")

with open(KG_JSON_PATH, "r", encoding="utf-8") as f:
    kg_edges = json.load(f)

print("✅ 三元组数量:", len(kg_edges))

entity_index = {}
relation_index = {}

for edge in kg_edges:
    h = edge["start_node"]["properties"]["name"]
    t = edge["end_node"]["properties"]["name"]
    r = edge["relation"]

    cid = edge["start_node"]["properties"].get("chunk id")
    end_label = edge["end_node"]["label"]

    if h not in entity_index:
        entity_index[h] = {
            "chunk_id": cid,
            "attributes": [],
            "relations": {}
        }

    if r == "has_attribute":
        entity_index[h]["attributes"].append(t)
    else:
        entity_index[h]["relations"].setdefault(r, []).append(t)
        relation_index.setdefault(r, []).append((h, t))

print("✅ 实体数量:", len(entity_index))
print("✅ 关系种类:", len(relation_index))

✅ 三元组数量: 20510
✅ 实体数量: 2352
✅ 关系种类: 432


In [3]:
from collections import Counter, defaultdict

# =========================
# 统计：每种关系的“具体条数”
# =========================
relation_counter = Counter()

for edge in kg_edges:
    r = edge["relation"]
    relation_counter[r] += 1

print("\n📊 各关系的三元组数量分布：")
for r, cnt in relation_counter.most_common():
    print(f"{r:<20} : {cnt}")


# =========================
# 统计：节点 Label 分布（start + end）
# =========================
start_label_counter = Counter()
end_label_counter = Counter()
all_label_counter = Counter()

for edge in kg_edges:
    s_label = edge["start_node"]["label"]
    e_label = edge["end_node"]["label"]

    start_label_counter[s_label] += 1
    end_label_counter[e_label] += 1

    all_label_counter[s_label] += 1
    all_label_counter[e_label] += 1

print("\n📊 Start Node Label 分布：")
for label, cnt in start_label_counter.most_common():
    print(f"{label:<15} : {cnt}")

print("\n📊 End Node Label 分布：")
for label, cnt in end_label_counter.most_common():
    print(f"{label:<15} : {cnt}")

print("\n📊 全部 Label 总分布（Start + End）：")
for label, cnt in all_label_counter.most_common():
    print(f"{label:<15} : {cnt}")


# =========================
# 统计：entity → entity VS entity → attribute
# =========================
ee_cnt = 0   # entity -> entity
ea_cnt = 0   # entity -> attribute
other_cnt = 0

for edge in kg_edges:
    s_label = edge["start_node"]["label"]
    e_label = edge["end_node"]["label"]

    if s_label == "entity" and e_label == "entity":
        ee_cnt += 1
    elif s_label == "entity" and e_label == "attribute":
        ea_cnt += 1
    else:
        other_cnt += 1

print("\n🔎 关系类型结构分布：")
print("entity -> entity    :", ee_cnt)
print("entity -> attribute :", ea_cnt)
print("other               :", other_cnt)


# =========================
# 统计：每个实体的“出边数量”（找高连接实体）
# =========================
degree_counter = Counter()

for edge in kg_edges:
    h = edge["start_node"]["properties"]["name"]
    degree_counter[h] += 1

top_entities = degree_counter.most_common(10)

print("\n⭐ 出边最多的 Top-10 实体（核心枢纽节点）：")
for name, cnt in top_entities:
    print(f"{name:<25} : {cnt}")


# =========================
# 统计：每个实体的“关系种类数”
# =========================
relation_type_per_entity = {}

for h, info in entity_index.items():
    relation_type_per_entity[h] = len(info["relations"])

top_relation_rich = sorted(
    relation_type_per_entity.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("\n🧠 关系类型最丰富的 Top-10 实体：")
for name, cnt in top_relation_rich:
    print(f"{name:<25} : {cnt}")



📊 各关系的三元组数量分布：
has_attribute        : 11020
member_of            : 1942
属于                   : 1114
使用                   : 813
介绍                   : 663
依赖于                  : 492
适用于                  : 471
包含                   : 437
represented_by       : 422
kw_filter_by         : 422
keyword_of           : 422
具有上界                 : 316
包含反例                 : 202
前置知识                 : 182
用于                   : 143
具有下界                 : 129
提供                   : 57
输入                   : 42
实现                   : 40
参演                   : 32
是                    : 27
具有                   : 26
基于                   : 24
导致                   : 22
输出                   : 21
具有评分                 : 21
解决                   : 20
与                    : 20
被评分                  : 20
描述                   : 19
支持                   : 19
不适用于                 : 16
应用于                  : 15
影响                   : 15
调用                   : 15
覆盖                   : 13
表示                   : 13
邻接